

## 1. The Core Idea

BPE was originally a data compression algorithm. In NLP, it’s adapted to find the most frequent **pairs of characters or subwords** and merge them into new tokens.

* Start with text split into characters.
* Repeatedly merge the **most frequent adjacent pair** into a new symbol.
* Continue until the vocabulary reaches the desired size.

This way, frequent words (like *“learning”*) stay whole, while rare/long words are broken into smaller subwords.

---

## 2. Why BPE for Tokenization?

* **Balances word-level and character-level tokenization**

  * Frequent words become single tokens (efficient).
  * Rare words break into smaller units (coverage).
* **Open-vocabulary handling**

  * Any unseen word can still be represented using subwords.
* **Language-agnostic**

  * Works well across English, Chinese, German, etc.

---

## 3. High-Level Steps of BPE

1. **Initialize Vocabulary**: Start with all characters as tokens.
   Example:
   Text = `"low lower lowest"`
   Initial tokens = `[l, o, w, e, r, s, t, <space>]`

2. **Tokenize Corpus into Characters**
   `"low"` → `["l", "o", "w"]`
   `"lower"` → `["l", "o", "w", "e", "r"]`

3. **Count Pair Frequencies**
   Find most common adjacent pair in the entire corpus.
   Example: `(l, o)`, `(o, w)`, `(w, e)` etc.

4. **Merge Most Frequent Pair**
   If `(l, o)` is most frequent → merge into `"lo"`.
   `"low"` → `["lo", "w"]`
   `"lower"` → `["lo", "w", "e", "r"]`

5. **Repeat Until Vocab Size Reached**
   Keep merging until you reach the predefined vocab size (e.g., 30k tokens).

---

## 4. Simple Example Walkthrough

Corpus: `"low lower lowest"`

**Step 1: Start with characters**
`["l", "o", "w"] ["l", "o", "w", "e", "r"] ["l", "o", "w", "e", "s", "t"]`

**Step 2: Count pairs**

* (l, o) → 3
* (o, w) → 3
* (w, e) → 2
* (e, r) → 1
* (e, s) → 1

**Step 3: Merge most frequent**

* Merge (l, o) → `"lo"`
  `["lo", "w"] ["lo", "w", "e", "r"] ["lo", "w", "e", "s", "t"]`

**Step 4: Count again**

* (lo, w) → 3
* (w, e) → 2

**Step 5: Merge again**

* Merge (lo, w) → `"low"`
  `["low"] ["low", "e", "r"] ["low", "e", "s", "t"]`

Now we have tokens: `"low", "er", "est"`

→ Vocabulary learned: `["l", "o", "w", "e", "r", "s", "t", "lo", "low", "er", "est"]`

---

## 5. Key Concepts to Remember

* **Frequency-driven** → merges are based on most common pairs.
* **Deterministic** → given same corpus + vocab size → same merges.
* **Subword efficiency** → rare words broken, frequent words whole.
* **OOV handling** → no unknown words, since any word can be decomposed into chars.



# Code Implementation


## Step 1: Prepare Corpus

We’ll use a small toy dataset so it’s easy to follow.

In [1]:
corpus = [
    "low",
    "lower",
    "lowest"
]


## Step 2: Represent Each Word as Characters + Special End Symbol

We add `</w>` (end-of-word) so the model knows word boundaries.

In [2]:
tokens = [list(word) + ["</w>"] for word in corpus]
print(tokens)

[['l', 'o', 'w', '</w>'], ['l', 'o', 'w', 'e', 'r', '</w>'], ['l', 'o', 'w', 'e', 's', 't', '</w>']]



**Output:**



```
[['l', 'o', 'w', '</w>'],
 ['l', 'o', 'w', 'e', 'r', '</w>'],
 ['l', 'o', 'w', 'e', 's', 't', '</w>']]


## Step 3: Build Vocabulary with Frequencies

In [3]:
from collections import Counter


def build_vocab(tokens):
    vocab = Counter([" ".join(token) for token in tokens])
    return vocab

vocab = build_vocab(tokens)
print(vocab)

Counter({'l o w </w>': 1, 'l o w e r </w>': 1, 'l o w e s t </w>': 1})


**Output:**

```
Counter({
 'l o w </w>': 1,
 'l o w e r </w>': 1,
 'l o w e s t </w>': 1
})
```


## Step 4: Count Pair Frequencies

We need to find the **most frequent pair** across the vocabulary.

In [4]:
def get_stats(vocab):
    pairs = Counter()
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols)-1):
            pairs[(symbols[i], symbols[i+1])] += freq
    return pairs

pairs = get_stats(vocab)
print(pairs)

Counter({('l', 'o'): 3, ('o', 'w'): 3, ('w', 'e'): 2, ('w', '</w>'): 1, ('e', 'r'): 1, ('r', '</w>'): 1, ('e', 's'): 1, ('s', 't'): 1, ('t', '</w>'): 1})


**Output (example):**

```
Counter({
 ('l', 'o'): 3,
 ('o', 'w'): 3,
 ('w', '</w>'): 1,
 ('w', 'e'): 2,
 ('e', 'r'): 1,
 ('e', 's'): 1,
 ('s', 't'): 1,
 ('t', '</w>'): 1
})
```


## Step 5: Merge Most Frequent Pair

In [5]:
def merge_vocab(pair, vocab):
    new_vocab = {}
    bigram = " ".join(pair)
    replacement = "".join(pair)
    for word in vocab:
        new_word = word.replace(bigram, replacement)
        new_vocab[new_word] = vocab[word]
    return new_vocab


## Step 6: Run BPE Iteratively

We’ll run a fixed number of merges (say, 10) to grow the vocabulary.

In [6]:
vocab = build_vocab(tokens)

num_merges = 10
for i in range(num_merges):
    pairs = get_stats(vocab)
    if not pairs:
        break
    best = max(pairs, key=pairs.get)
    vocab = merge_vocab(best, vocab)
    print(f"Step {i+1}: Merged {best}")
    print(vocab)
    print("-"*40)

Step 1: Merged ('l', 'o')
{'lo w </w>': 1, 'lo w e r </w>': 1, 'lo w e s t </w>': 1}
----------------------------------------
Step 2: Merged ('lo', 'w')
{'low </w>': 1, 'low e r </w>': 1, 'low e s t </w>': 1}
----------------------------------------
Step 3: Merged ('low', 'e')
{'low </w>': 1, 'lowe r </w>': 1, 'lowe s t </w>': 1}
----------------------------------------
Step 4: Merged ('low', '</w>')
{'low</w>': 1, 'lowe r </w>': 1, 'lowe s t </w>': 1}
----------------------------------------
Step 5: Merged ('lowe', 'r')
{'low</w>': 1, 'lower </w>': 1, 'lowe s t </w>': 1}
----------------------------------------
Step 6: Merged ('lower', '</w>')
{'low</w>': 1, 'lower</w>': 1, 'lowe s t </w>': 1}
----------------------------------------
Step 7: Merged ('lowe', 's')
{'low</w>': 1, 'lower</w>': 1, 'lowes t </w>': 1}
----------------------------------------
Step 8: Merged ('lowes', 't')
{'low</w>': 1, 'lower</w>': 1, 'lowest </w>': 1}
----------------------------------------
Step 9: Merged 

### Example Output

```
Step 1: Merged ('l', 'o')
{'lo w </w>': 1, 'lo w e r </w>': 1, 'lo w e s t </w>': 1}

Step 2: Merged ('lo', 'w')
{'low </w>': 1, 'low e r </w>': 1, 'low e s t </w>': 1}

Step 3: Merged ('low', '</w>')
{'low</w>': 1, 'low e r </w>': 1, 'low e s t </w>': 1}

Step 4: Merged ('e', 'r')
{'low</w>': 1, 'low er </w>': 1, 'low e s t </w>': 1}

Step 5: Merged ('low', 'er')
{'low</w>': 1, 'lower </w>': 1, 'low e s t </w>': 1}

Step 6: Merged ('e', 's')
{'low</w>': 1, 'lower </w>': 1, 'low es t </w>': 1}

Step 7: Merged ('es', 't')
{'low</w>': 1, 'lower </w>': 1, 'lowest </w>': 1}
```



✅ Now you can see how words like `"lower"` and `"lowest"` were gradually built from character-level tokens into meaningful subword units.

## Wrap Up
1. Builds a final vocabulary list (token → ID)

2. Provides encode() / decode() functions to convert text to tokens and back?


## 🔹 Step 1: Train BPE and Build Vocabulary
We’ll wrap the whole training loop into a class.

In [7]:

from collections import Counter

class BPETokenizer:
    def __init__(self, num_merges=50):
        self.num_merges = num_merges
        self.vocab = {}
        self.bpe_merges = []
        self.token2id = {}
        self.id2token = {}

    def build_vocab(self, corpus):
        tokens = [list(word) + ["</w>"] for word in corpus]
        vocab = Counter([" ".join(token) for token in tokens])
        return vocab

    def get_stats(self, vocab):
        pairs = Counter()
        for word, freq in vocab.items():
            symbols = word.split()
            for i in range(len(symbols)-1):
                pairs[(symbols[i], symbols[i+1])] += freq
        return pairs

    def merge_vocab(self, pair, vocab):
        new_vocab = {}
        bigram = " ".join(pair)
        replacement = "".join(pair)
        for word in vocab:
            new_word = word.replace(bigram, replacement)
            new_vocab[new_word] = vocab[word]
        return new_vocab

    def train(self, corpus):
        vocab = self.build_vocab(corpus)

        for i in range(self.num_merges):
            pairs = self.get_stats(vocab)
            if not pairs:
                break
            best = max(pairs, key=pairs.get)
            vocab = self.merge_vocab(best, vocab)
            self.bpe_merges.append(best)

        # Build final token vocabulary
        tokens = set()
        for word in vocab:
            tokens.update(word.split())
        tokens = sorted(tokens)

        self.token2id = {tok: idx for idx, tok in enumerate(tokens)}
        self.id2token = {idx: tok for tok, idx in self.token2id.items()}


## 🔹 Step 2: Encode Function
Applies the learned BPE merges to a new word.

In [8]:

def apply_bpe(self, word):
        # Start with characters + </w>
        symbols = list(word) + ["</w>"]

        # Iteratively apply merges
        for merge in self.bpe_merges:
            i = 0
            while i < len(symbols) - 1:
                if (symbols[i], symbols[i+1]) == merge:
                    symbols[i:i+2] = ["".join(merge)]
                else:
                    i += 1
        return symbols

def encode(self, text):
        words = text.split()
        tokens = []
        ids = []
        for word in words:
            subwords = self.apply_bpe(word)
            tokens.extend(subwords)
            ids.extend([self.token2id[sub] for sub in subwords if sub in self.token2id])
        return tokens, ids


## 🔹 Step 3: Decode Function
Takes IDs back to words.

In [10]:

def decode(self, ids):
        tokens = [self.id2token[i] for i in ids]
        # Join and remove </w> markers
        text = "".join(tokens).replace("</w>", " ")
        return text.strip()


## 🔹 Step 4: Test the Tokenizer

In [ ]:

# Example corpus
corpus = ["low", "lower", "lowest"]

# Train tokenizer
bpe = BPETokenizer(num_merges=10)
bpe.train(corpus)

print("Final Vocabulary:", bpe.token2id)

# Encode
tokens, ids = bpe.encode("lowest lower")
print("Tokens:", tokens)
print("IDs:", ids)

# Decode
decoded = bpe.decode(ids)
print("Decoded:", decoded)

In [12]:
from collections import Counter

class BPETokenizer:
    def __init__(self, num_merges=50):
        self.num_merges = num_merges
        self.vocab = {}
        self.bpe_merges = []
        self.token2id = {}
        self.id2token = {}

    # Step 1: Build vocab from corpus
    def build_vocab(self, corpus):
        tokens = [list(word) + ["</w>"] for word in corpus]
        vocab = Counter([" ".join(token) for token in tokens])
        return vocab

    # Step 2: Count pair frequencies
    def get_stats(self, vocab):
        pairs = Counter()
        for word, freq in vocab.items():
            symbols = word.split()
            for i in range(len(symbols)-1):
                pairs[(symbols[i], symbols[i+1])] += freq
        return pairs

    # Step 3: Merge the most frequent pair
    def merge_vocab(self, pair, vocab):
        new_vocab = {}
        bigram = " ".join(pair)
        replacement = "".join(pair)
        for word in vocab:
            new_word = word.replace(bigram, replacement)
            new_vocab[new_word] = vocab[word]
        return new_vocab

    # Step 4: Train BPE
    def train(self, corpus):
        vocab = self.build_vocab(corpus)

        for i in range(self.num_merges):
            pairs = self.get_stats(vocab)
            if not pairs:
                break
            best = max(pairs, key=pairs.get)
            vocab = self.merge_vocab(best, vocab)
            self.bpe_merges.append(best)

        # Build final vocabulary
        tokens = set()
        for word in vocab:
            tokens.update(word.split())
        tokens = sorted(tokens)

        self.token2id = {tok: idx for idx, tok in enumerate(tokens)}
        self.id2token = {idx: tok for tok, idx in self.token2id.items()}

    # Step 5: Apply BPE to a single word
    def apply_bpe(self, word):
        symbols = list(word) + ["</w>"]
        for merge in self.bpe_merges:
            i = 0
            while i < len(symbols) - 1:
                if (symbols[i], symbols[i+1]) == merge:
                    symbols[i:i+2] = ["".join(merge)]
                else:
                    i += 1
        return symbols

    # Step 6: Encode text → tokens + IDs
    def encode(self, text):
        words = text.split()
        tokens = []
        ids = []
        for word in words:
            subwords = self.apply_bpe(word)
            tokens.extend(subwords)
            ids.extend([self.token2id[sub] for sub in subwords if sub in self.token2id])
        return tokens, ids

    # Step 7: Decode IDs → text
    def decode(self, ids):
        tokens = [self.id2token[i] for i in ids]
        text = "".join(tokens).replace("</w>", " ")
        return text.strip()


In [13]:
# Training data
corpus = ["low", "lower", "lowest"]

# Train tokenizer
bpe = BPETokenizer(num_merges=10)
bpe.train(corpus)

print("Final Vocabulary:", bpe.token2id)

# Encode
tokens, ids = bpe.encode("lowest lower")
print("Tokens:", tokens)
print("IDs:", ids)

# Decode
decoded = bpe.decode(ids)
print("Decoded:", decoded)


Final Vocabulary: {'low</w>': 0, 'lower</w>': 1, 'lowest</w>': 2}
Tokens: ['lowest</w>', 'lower</w>']
IDs: [2, 1]
Decoded: lowest lower
